# Project 1: Titanic Exploratory Data Analysis (EDA)

**Question:** Who survived the Titanic, and why?

In this project you will load a real dataset, clean it, ask questions, and answer them with charts.
That is the whole job of EDA, and it is the first step of every ML project.

**How to use this notebook**
- Run each cell in order with **Shift + Enter**.
- Read the short explanation above each cell before running it.
- Cells marked **YOUR TURN** are for you to write. Try them before asking for help.
- At the end, write your 5 findings in your own words.

**Dataset:** 891 passengers from the Titanic (1912). Each row is one person.

| Column | Meaning |
|---|---|
| Survived | 0 = died, 1 = survived (**our target**) |
| Pclass | Ticket class: 1 = first, 2 = second, 3 = third |
| Name, Sex, Age | Passenger details |
| SibSp | Number of siblings or spouses aboard |
| Parch | Number of parents or children aboard |
| Ticket, Fare | Ticket number and price paid |
| Cabin | Cabin number (often missing) |
| Embarked | Port boarded: C = Cherbourg, Q = Queenstown, S = Southampton |

## Step 0: Setup and load the data

`import` loads the libraries. `pd.read_csv` reads a CSV file (from your computer or a URL) into a **DataFrame**, which is a table like Excel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)

print("Loaded!", df.shape)

## Step 1: First look at the data

Always start with these 4 commands on any new dataset:
- `head()` shows the first rows
- `shape` shows (rows, columns)
- `info()` shows column types and how many values are missing
- `describe()` shows statistics for number columns

In [ ]:
df.head()

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
df.describe()

**Think about it:** Look at the `describe()` output.
- What is the average age? The oldest passenger?
- What fraction survived? (Hint: the mean of `Survived`, because it is 0s and 1s.)
- Fare has a very large max compared with its median (50%). What does that tell you?

## Step 2: Find missing values

Real data is messy. Before any analysis, find what is missing.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing": missing, "percent": missing_pct})

In [ ]:
# Visualize it
missing_pct[missing_pct > 0].plot(kind="bar", color="salmon")
plt.title("Percent of missing values per column")
plt.ylabel("% missing")
plt.show()

## Step 3: Clean the data

Three columns have missing values. Each needs a different fix:

1. **Cabin** (mostly missing): too empty to fill, but *whether* someone had a cabin number may matter. We create `HasCabin` (1 or 0).
2. **Age** (about 20% missing): fill with the **median age of people with the same sex and class**. This is smarter than one overall average.
3. **Embarked** (2 missing): fill with the most common port.

We work on a copy so the original stays untouched.

In [ ]:
data = df.copy()

# 1. Cabin -> HasCabin
data["HasCabin"] = data["Cabin"].notnull().astype(int)

# 2. Age -> median by Sex and Pclass
data["Age"] = data["Age"].fillna(
    data.groupby(["Sex", "Pclass"])["Age"].transform("median")
)

# 3. Embarked -> most common value
data["Embarked"] = data["Embarked"].fillna(data["Embarked"].mode()[0])

# Check: only Cabin should still have missing values
data.isnull().sum()

## Step 4: Feature engineering (create new useful columns)

Good analysts create columns that make patterns easier to see:
- **FamilySize** = siblings/spouses + parents/children + 1 (the person)
- **IsAlone** = 1 if travelling alone
- **AgeGroup** = age bucketed into ranges
- **Title** = Mr, Mrs, Miss, Master... pulled out of the Name column

In [ ]:
data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

data["AgeGroup"] = pd.cut(
    data["Age"],
    bins=[0, 12, 18, 30, 45, 60, 100],
    labels=["Child 0-12", "Teen 13-18", "Young adult 19-30",
            "Adult 31-45", "Middle age 46-60", "Senior 60+"],
)

# "Braund, Mr. Owen Harris" -> "Mr"
data["Title"] = data["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip()
rare = data["Title"].value_counts()[lambda s: s < 10].index
data["Title"] = data["Title"].replace(rare, "Rare")

data[["Name", "Title", "Age", "AgeGroup", "FamilySize", "IsAlone", "HasCabin"]].head(10)

## Step 5: Ask questions and answer them with charts

This is the heart of EDA. For each question: compute the number, then chart it.

### Q1: How many people survived overall?

In [ ]:
rate = data["Survived"].mean()
print(f"Overall survival rate: {rate:.1%}")

sns.countplot(data=data, x="Survived", hue="Survived", palette=["salmon", "seagreen"], legend=False)
plt.xticks([0, 1], ["Died", "Survived"])
plt.title("Survivors vs. deaths")
plt.show()

### Q2: Did women survive more than men? ("Women and children first")

In [ ]:
print(data.groupby("Sex")["Survived"].mean().round(3))

sns.barplot(data=data, x="Sex", y="Survived", hue="Sex", palette="Set2", legend=False)
plt.title("Survival rate by sex")
plt.ylabel("Survival rate")
plt.show()

### Q3: Did ticket class (wealth) matter?

In [ ]:
print(data.groupby("Pclass")["Survived"].mean().round(3))

sns.barplot(data=data, x="Pclass", y="Survived", hue="Pclass", palette=["#1f4e79", "#4f8fc0", "#9cc3e4"], legend=False)
plt.title("Survival rate by ticket class")
plt.xlabel("Class (1 = first)")
plt.ylabel("Survival rate")
plt.show()

### Q4: Sex and class together

Two factors at once often tell a bigger story. A **heatmap** shows the survival rate for every combination.

In [ ]:
pivot = data.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="mean")

sns.heatmap(pivot, annot=True, fmt=".0%", cmap="RdYlGn", vmin=0, vmax=1)
plt.title("Survival rate by sex and class")
plt.show()

### Q5: Were children more likely to survive?

In [ ]:
by_age = data.groupby("AgeGroup", observed=False)["Survived"].mean()
print(by_age.round(3))

by_age.plot(kind="bar", color="steelblue")
plt.title("Survival rate by age group")
plt.ylabel("Survival rate")
plt.xticks(rotation=30, ha="right")
plt.show()

In [ ]:
# Age distribution of survivors vs. non-survivors
sns.histplot(data=data, x="Age", hue="Survived", bins=30, kde=True,
             palette=["salmon", "seagreen"], element="step")
plt.title("Age distribution by survival (0 = died, 1 = survived)")
plt.show()

### Q6: Did the fare paid make a difference?

Fare is very skewed (a few people paid a lot), so a **box plot** on a log scale is easier to read.

In [ ]:
print(data.groupby("Survived")["Fare"].median())

sns.boxplot(data=data, x="Survived", y="Fare", hue="Survived", palette=["salmon", "seagreen"], legend=False)
plt.yscale("log")
plt.xticks([0, 1], ["Died", "Survived"])
plt.title("Fare paid (log scale) by survival")
plt.show()

### Q7: Did family size matter?

Write the code yourself. Follow the same pattern as Q3:
1. Group by `FamilySize` and take the mean of `Survived`.
2. Make a bar plot.
3. Then do the same for `IsAlone`.

**Hint:** `data.groupby("FamilySize")["Survived"].mean()`

In [ ]:
# Q7: Family size
by_family = data.groupby("FamilySize")["Survived"].agg(["mean", "count"])
print(by_family.round(3))
print()
print(data.groupby("IsAlone")["Survived"].mean().rename({0: "With family", 1: "Alone"}).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=data, x="FamilySize", y="Survived", color="mediumpurple", ax=axes[0])
axes[0].set_title("Survival rate by family size")
axes[0].set_ylabel("Survival rate")
sns.barplot(data=data, x="IsAlone", y="Survived", hue="IsAlone", palette="Set2", legend=False, ax=axes[1])
axes[1].set_xticks([0, 1], ["With family", "Alone"])
axes[1].set_xlabel("")
axes[1].set_title("Alone vs. with family")
axes[1].set_ylabel("Survival rate")
plt.tight_layout()
plt.show()


### Q8: Title and port

1. What is the survival rate for each `Title`? Which title survived most?
2. What is the survival rate for each `Embarked` port? Can you explain it using class? (Hint: `pd.crosstab(data["Embarked"], data["Pclass"])`)

In [ ]:
# Q8: Title and port
by_title = data.groupby("Title")["Survived"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(by_title.round(3))
print()
print(data.groupby("Embarked")["Survived"].mean().round(3))
print()
print("Class mix per port (share of passengers):")
print(pd.crosstab(data["Embarked"], data["Pclass"], normalize="index").round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=data, x="Title", y="Survived", order=by_title.index, color="teal", ax=axes[0])
axes[0].set_title("Survival rate by title")
axes[0].set_ylabel("Survival rate")
port_names = data["Embarked"].map({"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"})
sns.barplot(data=data.assign(Port=port_names), x="Port", y="Survived", hue="Pclass",
            palette=["#1f4e79", "#4f8fc0", "#9cc3e4"], ax=axes[1])
axes[1].set_title("Survival by port, split by class")
axes[1].set_ylabel("Survival rate")
plt.tight_layout()
plt.show()


### Q9: Which numeric features relate to survival? (Correlation)

Correlation goes from -1 to +1. Near 0 means no straight-line relationship.
This helps you pick features for the ML model you will build in Phase 2.

In [ ]:
num_cols = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "HasCabin"]
corr = data[num_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between numeric features")
plt.show()

print("Correlation with Survived:")
print(corr["Survived"].drop("Survived").sort_values(ascending=False).round(2))

## Step 6: Save the cleaned data

You will reuse this in Phase 2 to train your first survival prediction model.

In [ ]:
data.to_csv("titanic_clean.csv", index=False)
print("Saved titanic_clean.csv with", data.shape[0], "rows and", data.shape[1], "columns")

## Step 7 — YOUR TURN: Write your 5 key findings

This is the most important part. Recruiters read this, not the code.
Write each finding as **one sentence with a number**, then one sentence explaining *why*.

Example: *"Women survived at about X% versus Y% for men, likely because of the 'women and children first' rule for lifeboats."*

1. **Finding 1:**
2. **Finding 2:**
3. **Finding 3:**
4. **Finding 4:**
5. **Finding 5:**

**Limitations:** (for example: only 891 of about 2,200 people aboard; ages were filled in with medians)

**Next step:** In Phase 2 I will train a model to predict survival using these features.

## Bonus

- Make a single figure with 4 charts side by side using `plt.subplots(2, 2)`.
- Is there a "deck" pattern? Extract the first letter of `Cabin` (A, B, C...) and check survival by deck.
- Did children in 3rd class survive less than children in 1st class? Filter and compare.

In [ ]:
# Bonus: deck and children by class
data["Deck"] = data["Cabin"].str[0].fillna("Unknown")
print(data.groupby("Deck")["Survived"].agg(["mean", "count"]).round(3))

children = data[data["Age"] <= 12]
print()
print("Children (12 and under) survival by class:")
print(children.groupby("Pclass")["Survived"].agg(["mean", "count"]).round(3))
